# 02 — Download and normalize historical datasets\nThis notebook downloads three public source datasets into the Colab runtime, normalizes them, and writes an audit manifest. It does not commit raw data to GitHub.\n\nDefault window: 2016-11-17 through 2026-11-17. Source coverage is uneven: the club dataset states coverage through 2023, the men's international dataset through 2024, and the women's dataset is incomplete.\n\nImportant: national-team sources include scores that may include extra time. FIFA A status remains unknown unless a row is matched to the evidence-backed FIFA registry.

In [ ]:
!git clone https://github.com/cifvck06660/FOOTBALL-PROBABILITY-LAB.git\n%cd FOOTBALL-PROBABILITY-LAB\n!pip -q install -e ".[dev]"

In [ ]:
!python scripts/ingest_datasets.py --since 2016-11-17 --until 2026-11-17

In [ ]:
import json\nfrom pathlib import Path\nimport pandas as pd\n\nprocessed = Path('data/processed')\nmanifest = json.loads((processed / 'ingestion_manifest.json').read_text())\nprint(json.dumps(manifest['date_window_inclusive'], indent=2))\npd.DataFrame([\n    {'source': key, 'rows': value['normalized_rows_in_window'],\n     'model_eligible_90m': value['model_eligible_90m_rows'],\n     'fifa_a_verified': value['fifa_a_verified_rows']}\n    for key, value in manifest['sources'].items()\n])

In [ ]:
club = pd.read_csv(processed / 'club_matches_10y.csv')\nmen = pd.read_csv(processed / 'national_men_candidates_10y.csv')\nwomen = pd.read_csv(processed / 'national_women_candidates_10y.csv')\n\nprint('Club date range:', club['date'].min(), 'to', club['date'].max())\nprint('Men candidate date range:', men['date'].min(), 'to', men['date'].max())\nprint('Women candidate date range:', women['date'].min(), 'to', women['date'].max())\nprint('Men FIFA status:', men['fifa_a_match_status'].value_counts(dropna=False).to_dict())\nprint('Women level status:', women['team_level'].value_counts(dropna=False).to_dict())

## Review before modeling\nDo not fit the national-team 90-minute goal model from these candidate CSVs yet. First verify FIFA A status and obtain regulation-time scores. The club file is filtered to full_time = F.